
# GVH Diagonal Cubic 0.3.2.7.3.6.1 — Complete Norm-Constraint Chain and 4×4 Poisson Rank Correction

**Auteur :** Charlemagne O Laurince

Objectif : corriger `0.3.2.7.3.6` sur la branche générique
\[
c_{14}\neq0,\qquad c_{\rm time}\neq0,\qquad \Delta\neq0,
\]
en fermant
\[
p_\lambda\to\chi\to\psi\to\rho,
\]
puis en construisant la matrice de Poisson complète sur
\[
\Phi_A=(p_\lambda,\chi,\psi,\rho).
\]

Les branches \(c_{14}=0\) et \(c_{\rm time}=0\) sont **exclues de la substitution générique** car les expressions inversées sont singulières.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:

import sympy as sp, json
from pathlib import Path
print("SymPy:", sp.__version__)


SymPy: 1.14.0



## 1. Hamiltonien local générique et contrainte de norme
\[
H_{\rm kin}
=
-\frac{p_s^2}{4c_{\rm time}}
+
\frac{p_i p_i}{4c_{14}},
\qquad
\chi=-s^2+v^2+1.
\]
Avec
\[
H_{\rm loc}=H_{\rm kin}-\lambda_{\rm mult}\chi.
\]


In [2]:

c14,ct = sp.symbols("c14 c_time", nonzero=True, real=True)
s,v1,v2,v3 = sp.symbols("s v1 v2 v3", real=True)
ps,p1,p2s,p3 = sp.symbols("p_s p1 p2 p3", real=True)
lam,plam = sp.symbols("lambda_mult p_lambda", real=True)

vnorm2 = v1**2+v2**2+v3**2
pnorm2 = p1**2+p2s**2+p3**2
chi = -s**2+vnorm2+1
Hkin = -ps**2/(4*ct)+pnorm2/(4*c14)
Hloc = Hkin-lam*chi


In [3]:

q=[s,v1,v2,v3,lam]
mom=[ps,p1,p2s,p3,plam]
def PB(F,G):
    return sp.expand(sum(sp.diff(F,qa)*sp.diff(G,pa)-sp.diff(F,pa)*sp.diff(G,qa)
                         for qa,pa in zip(q,mom)))

assert PB(s,ps)==1
assert PB(lam,plam)==1
print("Poisson convention: PASS")


Poisson convention: PASS



## 2. Chaîne \(p_\lambda\to\chi\to\psi\)

\[
\psi=\{\chi,H_{\rm loc}\}
=
\frac{s\,p_s}{c_{\rm time}}
+
\frac{v^ip_i}{c_{14}},
\]

\[
\Delta=\{\chi,\psi\}
=
2\left(\frac{v^2}{c_{14}}-\frac{s^2}{c_{\rm time}}\right).
\]


In [4]:

assert sp.simplify(PB(plam,Hloc)-chi)==0
psi = sp.factor(PB(chi,Hloc))
Delta = sp.factor(PB(chi,psi))

psi_expected = s*ps/ct + (v1*p1+v2*p2s+v3*p3)/c14
Delta_expected = 2*(vnorm2/c14-s**2/ct)

assert sp.simplify(psi-psi_expected)==0
assert sp.simplify(Delta-Delta_expected)==0
print("psi =", psi)
print("Delta =", Delta)


psi = (c14*p_s*s + c_time*p1*v1 + c_time*p2*v2 + c_time*p3*v3)/(c14*c_time)
Delta = -2*(c14*s**2 - c_time*v1**2 - c_time*v2**2 - c_time*v3**2)/(c14*c_time)



## 3. Correction de \(\mathcal A\) et nouvelle contrainte \(\rho\)

\[
\mathcal A=\{\psi,H_{\rm kin}\}
=
-\frac{p_s^2}{2c_{\rm time}^2}
+
\frac{p_i p_i}{2c_{14}^2}.
\]

\[
\boxed{\rho=\mathcal A+\lambda_{\rm mult}\Delta\approx0.}
\]


In [5]:

Akin = sp.factor(PB(psi,Hkin))
Aexpected = -ps**2/(2*ct**2)+pnorm2/(2*c14**2)
assert sp.simplify(Akin-Aexpected)==0

rho = sp.factor(PB(psi,Hloc))
assert sp.simplify(rho-(Akin+lam*Delta))==0
assert sp.simplify(PB(plam,rho)+Delta)==0

print("A =", Akin)
print("rho =", rho)
print("{p_lambda,rho} =", sp.factor(PB(plam,rho)))


A = (-c14**2*p_s**2 + c_time**2*p1**2 + c_time**2*p2**2 + c_time**2*p3**2)/(2*c14**2*c_time**2)
rho = (-4*c14**2*c_time*lambda_mult*s**2 - c14**2*p_s**2 + 4*c14*c_time**2*lambda_mult*v1**2 + 4*c14*c_time**2*lambda_mult*v2**2 + 4*c14*c_time**2*lambda_mult*v3**2 + c_time**2*p1**2 + c_time**2*p2**2 + c_time**2*p3**2)/(2*c14**2*c_time**2)
{p_lambda,rho} = 2*(c14*s**2 - c_time*v1**2 - c_time*v2**2 - c_time*v3**2)/(c14*c_time)



## 4. Matrice 4×4 et rang générique

Pour
\[
\Phi_A=(p_\lambda,\chi,\psi,\rho),
\]
la matrice est antisymétrique.

La structure impose
\[
\boxed{\det C_4=\Delta^4}.
\]

Donc, pour \(\Delta\neq0\),
\[
\boxed{\operatorname{rank}C_4=4}.
\]


In [6]:

x = sp.factor(PB(chi,rho))
y = sp.factor(PB(psi,rho))
C4 = sp.Matrix([
    [0,       0,      0,      -Delta],
    [0,       0,      Delta,   x],
    [0,      -Delta,  0,       y],
    [Delta,  -x,     -y,       0],
])
assert C4 + C4.T == sp.zeros(4)
det_struct = sp.factor(C4.det())
assert sp.simplify(det_struct-Delta**4)==0

d=sp.symbols("d", nonzero=True)
xs,ys=sp.symbols("x y")
C4_generic=sp.Matrix([
    [0,0,0,-d],
    [0,0,d,xs],
    [0,-d,0,ys],
    [d,-xs,-ys,0],
])
assert C4_generic.rank()==4

print("det(C4) =", det_struct)
print("generic rank =", C4_generic.rank())


det(C4) = 16*(c14*s**2 - c_time*v1**2 - c_time*v2**2 - c_time*v3**2)**4/(c14**4*c_time**4)
generic rank = 4



## 5. Classification de Dirac corrigée

\[
\boxed{
N_{\rm first}^{(\rm norm)}=0,\qquad
N_{\rm second}^{(\rm norm)}=4.
}
\]

Rang \(4\) signifie quatre contraintes seconde classe, soit deux paires.


In [7]:

def dirac_dof(Nphase,Nfirst,Nsecond):
    return sp.Rational(Nphase-2*Nfirst-Nsecond,2)

local_dof=dirac_dof(10,0,4)
assert local_dof==3
print("Local directional DOF =",local_dof)


Local directional DOF = 3



## 6. Comptage total DC-4 conditionnel

\(15\) variables de configuration donnent

\[
\boxed{N_{\rm phase}=30}.
\]

Si les huit contraintes ADM sont finalement première classe :

\[
N_{\rm first}=8,\qquad N_{\rm second}=4,
\]

alors

\[
\boxed{
N_{\rm phys}^{DC4}=5
}
\]

reste un résultat conditionnel.


In [8]:

Nconfig=15
Nphase=2*Nconfig
conditional_total_dof=dirac_dof(Nphase,8,4)
assert Nphase==30
assert conditional_total_dof==5
print("N_phase total =",Nphase)
print("Conditional DC-4 DOF =",conditional_total_dof)


N_phase total = 30
Conditional DC-4 DOF = 5



## 7. Branches spéciales

- \(\Delta=0\) avec \(c_{14},c_{\rm time}\neq0\) : **OPEN**, audit séparé requis.
- \(c_{14}=0\) : redériver depuis le Lagrangien/Hessien avant inversion de Legendre.
- \(c_{\rm time}=0\) : même règle.

Aucun rang n'est attribué par substitution singulière.


In [9]:

branch_status = {
    "GENERIC":"PASS_rank4_if_Delta_nonzero",
    "Delta_zero":"OPEN_SEPARATE_DIRAC_AUDIT",
    "c14_zero":"REDERIVE_BEFORE_LEGENDRE_INVERSION",
    "c_time_zero":"REDERIVE_BEFORE_LEGENDRE_INVERSION",
}
branch_status


{'GENERIC': 'PASS_rank4_if_Delta_nonzero',
 'Delta_zero': 'OPEN_SEPARATE_DIRAC_AUDIT',
 'c14_zero': 'REDERIVE_BEFORE_LEGENDRE_INVERSION',
 'c_time_zero': 'REDERIVE_BEFORE_LEGENDRE_INVERSION'}


## 8. Verdict

\[
\boxed{
p_\lambda\to\chi\to\psi\to\rho
}
\]

est fermé sur la branche générique.

\[
\boxed{\det C_4=\Delta^4,\qquad \operatorname{rank}C_4=4}
\]

pour \(\Delta\neq0\).

\[
\boxed{N_{\rm phys}^{(u,\rm local)}=3}
\]

et

\[
\boxed{N_{\rm phys}^{DC4}=5}
\]

reste conditionnel à l'algèbre hypersurface.

\[
\boxed{\text{PASS CORRECTION / FULL FIELD STILL BLOCKED}}
\]

\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]


In [10]:

FINAL_STATUS="PASS-CORRECTION-GENERIC-NORM-CHAIN-4X4-RANK4_FULL-FIELD-BLOCKED"
DISPERSION_READY=False

artifact={
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.6.1",
    "final_status":FINAL_STATUS,
    "generic":{
        "Akin":str(Akin),
        "rho":str(rho),
        "Delta":str(Delta),
        "det_C4":"Delta^4",
        "rank_C4":4,
        "N_first_norm":0,
        "N_second_norm":4,
        "local_directional_DOF":3,
    },
    "total_DC4":{
        "N_configuration":15,
        "N_phase":30,
        "conditional_N_first_ADM":8,
        "N_second_norm":4,
        "conditional_DOF":5,
    },
    "branches":branch_status,
    "dispersion_ready":False,
    "next":"GVH_Diagonal_Cubic_0.3.2.7.3.7_Full_Coupled_Kinetic_Inversion_and_Hypersurface_Deformation_Algebra_Audit.ipynb"
}
export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path.cwd()/"gvh_exports"
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path=export_dir/"gvh_0.3.2.7.3.6.1_verdict.json"
artifact_path.write_text(json.dumps(artifact,indent=2),encoding="utf-8")

assert DISPERSION_READY is False
print("FINAL STATUS:",FINAL_STATUS)
print("Artifact:",artifact_path)


FINAL STATUS: PASS-CORRECTION-GENERIC-NORM-CHAIN-4X4-RANK4_FULL-FIELD-BLOCKED
Artifact: /content/gvh_exports/gvh_0.3.2.7.3.6.1_verdict.json
